In [1]:
import requests
import pandas as pd

url = 'https://apis.data.go.kr/1160100/service/GetGeneralProductInfoService/getCertifiedEmissionReductionPriceInfo'
params = {
    'serviceKey': '8f60b786617639fe6fd980f7708756a806a7f6a67fd149377d329a8ee028877d',
    'resultType': 'json',
    'pageNo': '1',
    'numOfRows': '4000',      # 여러 종목이 섞여있으므로 행 수를 충분히 크게 잡습니다
    'beginBasDt': '20210101',
    'endBasDt': '20211231'
}

response = requests.get(url, params=params)
data = response.json()
items = data['response']['body']['items']['item']
df = pd.DataFrame(items)

# 1. 핵심: 표준 배출권인 'KAU21'만 필터링합니다.
# (연도별로 조회한다면 KAU + 연도 뒷자리 형식을 찾으면 됩니다)
df = df[df['itmsNm'] == 'KAU21'].copy()

# 2. 데이터 타입 변환
df['basDt'] = pd.to_datetime(df['basDt'])
df['clpr'] = pd.to_numeric(df['clpr'])

# 3. 날짜순 정렬 후 월별 마지막 거래일 추출
df = df.sort_values('basDt')
monthly_last_trade = df.groupby(df['basDt'].dt.to_period('M')).last()

print("=== 2021년 KAU21 월별 마지막 거래일 종가 ===")
print(monthly_last_trade[['basDt', 'itmsNm', 'clpr']].reset_index(drop=True))

=== 2021년 KAU21 월별 마지막 거래일 종가 ===
        basDt itmsNm   clpr
0  2021-01-29  KAU21  20100
1  2021-02-26  KAU21  18900
2  2021-03-31  KAU21  20150
3  2021-04-30  KAU21  19900
4  2021-05-31  KAU21  19300
5  2021-06-30  KAU21  16150
6  2021-07-30  KAU21  20850
7  2021-08-31  KAU21  28000
8  2021-09-30  KAU21  30500
9  2021-10-29  KAU21  30400
10 2021-11-30  KAU21  33600
11 2021-12-30  KAU21  35100
